# M19a - Core validation-safe-region transfer

**Author:** Ildefons Magrans de Abril  
**Affiliation:** Universitat Politècnica de Catalunya - BarcelonaTech (UPC)

**Submission evidence.** This notebook executes the current frozen protocol used by the manuscript. Its generated values, not archived development-era summaries, are the submission evidence.

In [1]:
from pathlib import Path
import sys, numpy as np, pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import tcr_core as tcr
REPRO=ROOT/'results'/'reproduced'; REPRO.mkdir(parents=True,exist_ok=True)
def bootstrap_mean(x,n_boot=30000,seed=1):
    x=np.asarray(x,float); x=x[np.isfinite(x)]
    rng=np.random.default_rng(seed); idx=rng.integers(0,len(x),size=(n_boot,len(x)))
    b=x[idx].mean(axis=1)
    return float(x.mean()),float(np.quantile(b,.025)),float(np.quantile(b,.975))

In [2]:
SEED=20260621
TRIALS=12
TASKS=['controlled_d20_white_plus_distractor','memory_d10','narma10','lorenz_x']
CONFIG=dict(N=60,K=13,lengths=(1200,500,500),washout=100,input_scale=.8,ridge=1e-5)
rep=tcr.run_panel(TASKS,TRIALS,['temperature'],seed=SEED,**CONFIG)
rep.to_csv(REPRO/'m19a_replication_case_metrics.csv',index=False)
ci=bootstrap_mean(rep.safe_gain,seed=1901)
summary=pd.DataFrame([{'n_cases':len(rep),'near_optimal_containment':rep.near_contained.mean(),'exact_oracle_containment':rep.exact_contained.mean(),'mean_safe_gain':ci[0],'safe_gain_ci_low':ci[1],'safe_gain_ci_high':ci[2],'mean_full_grid_gain':rep.full_gain.mean(),'mean_safe_width':rep.safe_width.mean()}])
summary.to_csv(REPRO/'m19a_replication_summary.csv',index=False)
display(summary.round(6))

,n_cases,near_optimal_containment,exact_oracle_containment,mean_safe_gain,safe_gain_ci_low,safe_gain_ci_high,mean_full_grid_gain,mean_safe_width
0,48,0.791667,0.604167,0.001313,0.000508,0.002392,0.007873,3.020833
